# Week 4 — Fine-Tune TrOCR on the Urdu OCR Dataset

This picks up where Week 3 left off: `labels.csv` and the generated images from Week 3 get fine-tuned into `microsoft/trocr-base-printed` (Microsoft's pretrained OCR model), then evaluated on a held-out test set.

**Before running anything:** `Runtime > Change runtime type > GPU`. CPU training will run but takes hours instead of minutes.

A note on the attached `TensorFlow_with_GPU.ipynb`: that notebook's GPU-connectivity check is TensorFlow-specific (`tf.test.gpu_device_name()`). This project's stack is PyTorch + Hugging Face Transformers (as in the Week 4 handout), so the equivalent check below uses `torch.cuda.is_available()` instead — same idea, different library. Mixing TensorFlow calls into a PyTorch pipeline wouldn't do anything useful here, so it isn't used directly.

**What follows the official handout, and what's added on top:**
- Steps 1–4 below match the Week 4 handout's structure and step numbers.
- **Step 0** is new — it isn't in the handout, but a fresh Colab runtime starts empty, so it has to exist to reconnect to Week 3's data.
- Sections marked **Extra** are additions beyond the handout: a training-loss chart, Character/Word Error Rate alongside exact-match accuracy, automatic extraction of the worst predictions, and a visual grid of sample predictions.
- Three real bugs get fixed along the way (verified in a sandboxed test against `transformers==4.57.6`, the version pinned in Week 3): `from transformers import AdamW` no longer exists in this version and raises `ImportError` — `torch.optim.AdamW` is the fix; the labels Week 3 produced pad with the literal pad-token id instead of `-100`, so the loss function was never actually ignoring padding — fixed in the Dataset class below; and the handout's model name has a dropped hyphen (`trocr-baseprinted`) — corrected to `trocr-base-printed` to match Week 3 and the handout's own Sources section.


## Step 0: Reconnect to Your Week 3 Dataset

*(Not in the official handout.)* Week 3 ran in a GitHub Codespace and built up `labels.csv` plus generated images inside your repo. A fresh Colab runtime starts with an empty filesystem, so this cell gets that data back before anything else can run. Pick **one** of the two options below.

In [ ]:
import os
import pandas as pd

# OPTION A — clone your GitHub repo (use this if labels.csv + images are committed/pushed)
REPO_URL = "https://github.com/hamnasz/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran.git"  # TODO: replace with your repo URL

if not os.path.isdir("repo"):
    !git clone {REPO_URL} repo
else:
    print("repo/ already exists, skipping clone")

DATA_DIR = "repo/SI26-Week1/data"

# OPTION B — mount Google Drive instead (uncomment if that's where your data lives,
# and comment out the git clone block above)
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/<path-to-your-data>"  # TODO: replace with your path

LABELS_PATH = os.path.join(DATA_DIR, "labels.csv")
assert os.path.isfile(LABELS_PATH), (
    f"Couldn't find labels.csv at '{LABELS_PATH}'. Check REPO_URL and DATA_DIR above, "
    "and make sure Week 3's labels.csv + images were committed and pushed to GitHub "
    "(or switch to OPTION B if your data lives on Drive instead)."
)
print(f"Found labels.csv with {len(pd.read_csv(LABELS_PATH))} rows at {LABELS_PATH}")

## Step 1: Load the Pretrained TrOCR Model

TrOCR combines a vision encoder (reads the image) with a text decoder (outputs characters). This loads the version already trained on printed text, before fine-tuning it on the Urdu images from Week 3 — transfer learning.

In [ ]:
!pip install --upgrade "transformers==4.57.6" torch pillow pandas sentencepiece protobuf jiwer matplotlib --quiet

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU.")
    print("Training will still run on CPU, just far slower (hours instead of minutes).")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# Handout text has this as 'microsoft/trocr-baseprinted' (missing hyphen) -- that's not
# a real model on the Hub. The correct id (matching Week 3 and the handout's own Sources
# section) is:
MODEL_NAME = "microsoft/trocr-base-printed"

try:
    processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
    model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
except Exception as e:
    raise RuntimeError(
        f"Couldn't load '{MODEL_NAME}' from Hugging Face Hub: {e}. "
        "Check your internet connection and that huggingface.co is reachable."
    ) from e

model = model.to(device)

# Configure model for generation (standard TrOCR fine-tuning setup)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print("Model loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 2: Set Up Training

A `DataLoader` wraps the dataset and feeds it to the model in small batches — batch size 4 means the model sees 4 images at a time before updating its weights. The optimiser (`AdamW`) controls how the model adjusts itself after each batch.

The `UrduOCRDataset` class from Week 3 is redefined here so this notebook can run standalone in a fresh Colab session — skip this cell if Week 3's version is already in memory. One change from Week 3: padded label positions are now set to `-100` instead of the pad-token id. PyTorch's `CrossEntropyLoss` (which the model uses internally) ignores `-100` by default but nothing else — so Week 3's version was training the model to predict padding too, which quietly hurts both the loss numbers and generation quality. This is the fix the official TrOCR fine-tuning guide uses.

In [ ]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

MAX_LENGTH = 128

class UrduOCRDataset(Dataset):
    """PyTorch Dataset over labels.csv, skipping rows with no image on disk.

    Each item returns TrOCR-ready pixel_values and tokenized labels, with
    padding positions set to -100 so the loss function ignores them.
    """

    def __init__(self, csv_path, processor, data_dir, max_length=MAX_LENGTH):
        data = pd.read_csv(csv_path)
        has_file = data["image"].apply(lambda p: os.path.isfile(os.path.join(data_dir, p)))
        n_missing = int((~has_file).sum())
        if n_missing:
            print(f"Skipping {n_missing} rows with no image on disk (check Step 0's DATA_DIR)")
        self.data = data[has_file].reset_index(drop=True)
        self.processor = processor
        self.data_dir = data_dir
        self.max_length = max_length
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = os.path.join(self.data_dir, row["image"])
        try:
            image = Image.open(image_path).convert("RGB")
        except (FileNotFoundError, OSError) as e:
            raise FileNotFoundError(f"Row {idx}: can't open '{image_path}' ({e})")

        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()
        labels = self.processor.tokenizer(
            str(row["text"]), padding="max_length", max_length=self.max_length, truncation=True
        ).input_ids
        # Ignore padding positions in the loss (see markdown note above)
        labels = [l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels]
        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}


dataset = UrduOCRDataset(LABELS_PATH, processor, data_dir=DATA_DIR)
train_size = int(0.8 * len(dataset))
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, len(dataset) - train_size]
)
print(f"Train: {len(train_dataset)}  Test: {len(test_dataset)}")

In [ ]:
# NOTE: the handout says `from transformers import AdamW` -- that raises ImportError on
# transformers==4.57.6 (verified). Transformers removed its own AdamW in favor of PyTorch's;
# torch.optim.AdamW is the standard drop-in replacement.
from torch.optim import AdamW

BATCH_SIZE = 4
LEARNING_RATE = 5e-5

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

print(f"Training batches per epoch: {len(train_loader)}")
print("Ready to train!")

In [ ]:
# This cell will take 20-40 minutes on GPU. Do not close your browser tab while it runs.
# Watch the loss number decrease -- that means the model is learning.
NUM_EPOCHS = 3
loss_history = []       # one entry per batch, across all epochs -- feeds the loss chart below
epoch_avg_losses = []

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 30)
    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        try:
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        except torch.cuda.OutOfMemoryError:
            print(f"  Batch {batch_idx}: ran out of GPU memory. Lower BATCH_SIZE above "
                  "and re-run from the DataLoaders cell.")
            raise

        total_loss += loss.item()
        loss_history.append(loss.item())
        if batch_idx % 10 == 0:
            print(f"  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    epoch_avg_losses.append(avg_loss)
    print(f"Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}")

print("\nTraining complete!")
print(f"Training loss went from {loss_history[0]:.4f} (first batch) to {loss_history[-1]:.4f} (last batch)")

### Step 2 Extra: Visualize Training Progress

*(Not in the handout.)* The submission asks for a screenshot of the training output showing loss decreasing — an actual chart makes that point far more clearly than a screenshot of scrolling console text, and it's this cell's output that best supports the required `'Training loss went from X to X'` line.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(loss_history, color="#4C72B0", linewidth=1.2)
batches_per_epoch = len(train_loader)
for e in range(1, NUM_EPOCHS):
    axes[0].axvline(e * batches_per_epoch, color="#999999", linestyle=":", linewidth=1)
axes[0].set_title("Training Loss per Batch")
axes[0].set_xlabel("Batch (cumulative across epochs)")
axes[0].set_ylabel("Loss")

axes[1].plot(range(1, NUM_EPOCHS + 1), epoch_avg_losses, marker="o", color="#C44E52")
axes[1].set_title("Average Loss per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Average Loss")
axes[1].set_xticks(range(1, NUM_EPOCHS + 1))
for i, v in enumerate(epoch_avg_losses):
    axes[1].annotate(f"{v:.3f}", (i + 1, v), textcoords="offset points", xytext=(0, 8), ha="center")

plt.tight_layout()
plt.savefig("training_loss.png", dpi=150)
plt.show()
print("Saved to training_loss.png -- attach this for the 'screenshot of training output' requirement.")

## Step 3: Evaluate Your Model

After training, this tests the model on images it has never seen — the test split from above. `model.eval()` turns off weight updates during this step.

Beyond the handout's exact-match accuracy, this also reports **Character Error Rate (CER)** and **Word Error Rate (WER)** — exact-match is strict (one wrong character anywhere in the line counts as fully wrong), while CER/WER are the standard OCR metrics and give a more graded sense of how close predictions are.

In [ ]:
import jiwer

model.eval()
print("=== Model Evaluation on Test Images ===\n")

results = []
with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id  # undo the training-time mask for decoding

        generated_ids = model.generate(pixel_values, max_length=MAX_LENGTH)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)

        for pred, actual in zip(generated_text, actual_text):
            pred, actual = pred.strip(), actual.strip()
            results.append({
                "predicted": pred,
                "actual": actual,
                "exact_match": pred == actual,
                "cer": jiwer.cer(actual, pred) if actual else None,
            })
            print(f"Predicted: {pred}")
            print(f"Actual: {actual}")
            print()

correct = sum(r["exact_match"] for r in results)
total = len(results)
accuracy = (correct / total) * 100 if total > 0 else 0

refs = [r["actual"] for r in results]
hyps = [r["predicted"] for r in results]
overall_cer = jiwer.cer(refs, hyps) if refs else float("nan")
overall_wer = jiwer.wer(refs, hyps) if refs else float("nan")

print(f"Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")
print(f"Character Error Rate (CER): {overall_cer:.3f}")
print(f"Word Error Rate (WER): {overall_wer:.3f}")

### Step 3 Extra: Automatically Find the Worst Predictions

*(Not in the handout.)* The handout asks you to note 3-5 examples where the model got it wrong for Week 5. This pulls the worst ones by CER automatically instead of scrolling through the printed output above by hand.

In [ ]:
print("=== Examples where the model got it wrong (for your Week 5 discussion points) ===\n")

wrong = [r for r in results if not r["exact_match"]]
wrong_sorted = sorted(wrong, key=lambda r: (r["cer"] if r["cer"] is not None else 0), reverse=True)

if not wrong_sorted:
    print("No mistakes on the test set (or the test set is very small -- check the count above).")
else:
    for i, r in enumerate(wrong_sorted[:5], 1):
        print(f"{i}. Predicted: {r['predicted']}")
        print(f"   Actual:    {r['actual']}")
        print(f"   CER: {r['cer']:.3f}" if r["cer"] is not None else "   CER: n/a")
        print()

### Step 3 Extra: Visualize Sample Predictions

*(Not in the handout.)* A grid of actual test images next to what the model predicted vs. the ground truth — green border for an exact match, red for a mismatch.

In [ ]:
import textwrap
import random
from PIL import ImageDraw, ImageFont

# Map results back to their source image paths. test_loader has shuffle=False, so it
# iterates test_dataset in the same order as test_dataset.indices -- verified in testing.
test_image_paths = [test_dataset.dataset.data.iloc[i]["image"] for i in test_dataset.indices]
assert len(results) == len(test_image_paths), (
    f"Got {len(results)} predictions but {len(test_image_paths)} test images -- "
    "re-run the evaluation cell above before this one."
)
for r, img_path in zip(results, test_image_paths):
    r["image"] = img_path


def show_prediction_grid(results, data_dir, n=9, cols=3, seed=0,
                          font_path=FONTS["nastaliq"] if "FONTS" in dir() else "fonts/NotoNastaliqUrdu.ttf",
                          thumb_size=230, caption_h=140, font_size=15):
    """Grid of sample test predictions: image + predicted vs. actual text.

    Captions are drawn with PIL using RTL shaping (direction='rtl', language='ur'),
    the same approach used for the Week 3 dataset-sample grid, since matplotlib
    cannot shape Arabic-script text on its own.
    """
    if not results:
        print("No results to show.")
        return
    random.seed(seed)
    sample = random.sample(results, k=min(n, len(results)))

    n_cols = min(cols, len(sample))
    n_rows = -(-len(sample) // n_cols)
    cell_w, cell_h = thumb_size, thumb_size + caption_h
    row_h = int(font_size * 2.1)

    grid_img = Image.new("RGB", (n_cols * cell_w, n_rows * cell_h), (255, 255, 255))
    font = ImageFont.truetype(font_path, font_size)
    label_font = ImageFont.load_default()

    for i, r in enumerate(sample):
        row_c, col_c = divmod(i, n_cols)
        thumb = Image.open(os.path.join(data_dir, r["image"])).convert("RGB")
        thumb.thumbnail((thumb_size - 10, thumb_size - 10))
        cell = Image.new("RGB", (cell_w, cell_h), (255, 255, 255))
        cell.paste(thumb, ((cell_w - thumb.width) // 2, (thumb_size - thumb.height) // 2))

        draw = ImageDraw.Draw(cell)
        y = thumb_size + 8
        for label, text, color in [("Pred:", r["predicted"], (30, 60, 150)), ("True:", r["actual"], (20, 20, 20))]:
            draw.text((8, y), label, font=label_font, fill=(120, 120, 120))
            y += 14
            wrapped = textwrap.wrap(str(text), width=22)[:1]
            line = wrapped[0] if wrapped else str(text)
            if len(str(text)) > 22:
                line = line + "…"
            bbox = draw.textbbox((0, 0), line, font=font, direction="rtl", language="ur")
            tw = bbox[2] - bbox[0]
            draw.text((min(cell_w - 8, tw + 8), y), line, font=font, fill=color,
                       direction="rtl", language="ur", anchor="ra")
            y += row_h

        border_color = (85, 168, 104) if r["exact_match"] else (196, 78, 82)
        draw.rectangle([0, 0, cell_w - 1, cell_h - 1], outline=border_color, width=4)
        grid_img.paste(cell, (col_c * cell_w, row_c * cell_h))

    plt.figure(figsize=(n_cols * 3.2, n_rows * 4.1))
    plt.imshow(grid_img)
    plt.axis("off")
    n_correct = sum(r["exact_match"] for r in sample)
    plt.title(f"Sample Predictions ({n_correct}/{len(sample)} exact match) — green = correct, red = mismatch")
    plt.tight_layout()
    plt.show()

show_prediction_grid(results, DATA_DIR, n=9)

## Step 4: Save Your Model

Colab sessions reset when closed. Saving to Google Drive means you can reload the model in Week 5 without retraining from scratch. This also saves the loss history and evaluation metrics alongside the model, so Week 5's discussion points don't depend on remembering numbers from this session.

In [ ]:
import json

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    raise RuntimeError(
        "google.colab isn't available -- this cell needs to run inside Google Colab to save to Drive."
    )

save_path = "/content/drive/MyDrive/SI26-urdu-ocr-model"
try:
    model.save_pretrained(save_path)
    processor.save_pretrained(save_path)

    metrics = {
        "accuracy_pct": accuracy,
        "cer": overall_cer,
        "wer": overall_wer,
        "loss_first_batch": loss_history[0],
        "loss_last_batch": loss_history[-1],
        "epoch_avg_losses": epoch_avg_losses,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
    }
    with open(os.path.join(save_path, "week4_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print(f"Model saved to Google Drive: {save_path}")
    print("You can load this model again next week without retraining")
except OSError as e:
    raise RuntimeError(f"Couldn't save to {save_path}: {e}. Check your Drive has free space and is mounted.") from e

After running this cell, open [drive.google.com](https://drive.google.com) and confirm the `SI26-urdu-ocr-model` folder exists before closing Colab.

## Summary & Submission Checklist

- **GitHub link to this notebook** (with training + evaluation code) — push this file to your repo
- **`My model accuracy is X%`** — printed in Step 3 (`Accuracy: X% (n/m correct)`)
- **`Training loss went from X to X`** — printed at the end of the training loop cell
- **Screenshot of training output showing loss decreasing** — the Step 2 Extra chart (`training_loss.png`) covers this directly
- **3–5 wrong examples for Week 5** — auto-generated in Step 3 Extra

CER/WER, the prediction grid, and `week4_metrics.json` aren't required by the handout, but are there if useful context for Week 5's discussion.

One more thing worth flagging: the task text says to submit by Friday, July 25 — that's already past as of today (Sunday, July 26). Worth a quick check with your mentor on whether the deadline moved or that's just leftover boilerplate.